# Road Sense: train strategy models

Trains the tire degradation, pit strategy and weather impact models from the GR Cup
race timing data and publishes them to Supabase Storage for the backend.

**Before running**
1. *Add Input*: attach the race dataset (files named `<track>_pit_data.csv`,
   `<track>_race_data.csv`, `<track>_weather_data.csv`, in any folder).
2. *Settings → Internet*: **on** (to install pinned packages and upload).
3. *Add-ons → Secrets*: add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY`
   (optionally `SUPABASE_MODELS_BUCKET`, default `road-sense-models`) and attach them.

Nothing here falls back to generated data: a model without enough real data is
reported as `insufficient_data` / `skipped` in the manifest and not published.

In [ ]:
# Code version to train with (branch, tag or commit of the repository)
REPO_URL = "https://github.com/Oluwaseyi89/toyota-hack-the-track.git"
REPO_REF = "main"
DATA_DIR = "/kaggle/input"   # searched recursively for the CSVs
UPLOAD = True                # False = train and inspect only

In [ ]:
!rm -rf /kaggle/working/road-sense
!git clone --quiet --depth 1 --branch {REPO_REF} {REPO_URL} /kaggle/working/road-sense
%cd /kaggle/working/road-sense/model-training-service

Install the **pinned** versions. They must match the backend's
`requirements.txt`, because the backend unpickles these models. Training runs in a
fresh subprocess afterwards, so no kernel restart is needed.

In [ ]:
!pip install --quiet -r requirements.txt
!python -c "import sys, sklearn, numpy; print(sys.version.split()[0], 'sklearn', sklearn.__version__, 'numpy', numpy.__version__)"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "SUPABASE_MODELS_BUCKET"):
    try:
        os.environ[name] = secrets.get_secret(name)
        print(f"{name}: set")
    except Exception:
        print(f"{name}: not set")

Train. The Python version is checked against the pin as well; if Kaggle's image uses
a different Python minor version, rerun with `--allow-version-mismatch` only if the
scikit-learn and numpy versions above match the pins (they matter for unpickling).

In [ ]:
upload_flag = "--upload" if UPLOAD else ""
!python -m road_sense_training.train --data-dir {DATA_DIR} {upload_flag}

In [ ]:
import json, pathlib

runs = sorted(pathlib.Path("/kaggle/working/road_sense_artifacts").glob("*/manifest.json"))
manifest = json.loads(runs[-1].read_text())
print("run:", manifest["run_id"], "| versions:", {k: manifest["versions"][k] for k in ("python", "scikit-learn", "numpy")})
for track in manifest["dataset"]["tracks"]:
    print(" ", track)
for name, model in manifest["models"].items():
    print(f"\n{name}: {model['status']} {model['reason']}")
    print("  metrics:", model["metrics"])